# debug_test

按当前主流程逐步调试：读取 VASP 数据、推断初末态极化、做 branch 矫正、计算矫顽场。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from coercive_field.branch import build_branch_images, continuous_center_zero_branch
from coercive_field.calculation import (
    analyze_path,
    candidate_indices_for_mode,
    coercive_field_from_requirements,
    coercive_requirements,
    infer_initial_polarization,
    is_non_180_switch,
    resolve_continuity_threshold,
    resolve_final_polarization,
)
from coercive_field.cli import default_samples
from coercive_field.directions import resolve_direction
from coercive_field.vasp_io import parse_structure, read_raw_images

In [ ]:
LABEL = "HfO2"
PATH = Path("Pmn2_1") / "xyz-101-000"
ENERGY_SOURCE = "oszicar"
FIELD_DIRECTION_ARG = "0,0,-1"
FIELD_COORDINATES = "lattice"
BARRIER_MODE = "full-path"
COERCIVE_CRITERION = "downhill"
INITIAL_POLARIZATION = "auto"
FINAL_POLARIZATION = "auto"
CONTINUITY_THRESHOLD = "auto"

In [ ]:
raw_images, notes = read_raw_images(LABEL, PATH, ENERGY_SOURCE)
reference_lattice, _ = parse_structure(raw_images[0].structure_path)
center_image = len(raw_images) // 2

pd.DataFrame(
    [
        {
            "image": image.image,
            "rel_energy_eV": image.rel_energy_eV,
            "volume_A3": image.volume_A3,
            "raw_P_x": image.raw_polarization_uC_cm2[0],
            "raw_P_y": image.raw_polarization_uC_cm2[1],
            "raw_P_z": image.raw_polarization_uC_cm2[2],
        }
        for image in raw_images
    ]
)

In [ ]:
continuity_threshold, used_auto_threshold = resolve_continuity_threshold(CONTINUITY_THRESHOLD, raw_images)
initial_p, initial_direction = infer_initial_polarization(
    INITIAL_POLARIZATION,
    raw_images,
    PATH,
    reference_lattice,
    center_image,
)
final_p = resolve_final_polarization(FINAL_POLARIZATION, initial_p, PATH)
endpoint_reference = final_p if is_non_180_switch(initial_p, final_p) else None
field_direction = resolve_direction(FIELD_DIRECTION_ARG, FIELD_COORDINATES, reference_lattice)

print("center_image =", center_image)
print("continuity_threshold =", continuity_threshold, "auto =", used_auto_threshold)
print("initial_direction =", initial_direction)
print("initial_p =", initial_p)
print("final_p =", final_p)
print("endpoint_reference =", endpoint_reference)
print("field_direction =", field_direction)
print("notes =", notes)

In [ ]:
raw_polarizations = [image.raw_polarization_uC_cm2 for image in raw_images]
quantum_vectors = [image.quantum_vectors_uC_cm2 for image in raw_images]
center_branch_pols, center_branch_indices = continuous_center_zero_branch(
    raw_polarizations,
    quantum_vectors,
    center_image,
)

pd.DataFrame(
    [
        {
            "image": raw_images[i].image,
            "n1": center_branch_indices[i][0],
            "n2": center_branch_indices[i][1],
            "n3": center_branch_indices[i][2],
            "P_x": center_branch_pols[i][0],
            "P_y": center_branch_pols[i][1],
            "P_z": center_branch_pols[i][2],
        }
        for i in range(len(raw_images))
    ]
)

In [ ]:
branch_images = build_branch_images(
    raw_images=raw_images,
    projection_direction=field_direction,
    center_image=center_image,
    continuity_threshold_uC_cm2=continuity_threshold,
    endpoint_reference=endpoint_reference,
)

pd.DataFrame(
    [
        {
            "image": image.image,
            "n1": image.branch_index[0],
            "n2": image.branch_index[1],
            "n3": image.branch_index[2],
            "P_x": image.branch_polarization_uC_cm2[0],
            "P_y": image.branch_polarization_uC_cm2[1],
            "P_z": image.branch_polarization_uC_cm2[2],
            "projection": image.branch_projection_uC_cm2,
            "jump": image.max_abs_component_jump_from_prev_uC_cm2,
            "continuous": image.component_continuity_ok,
        }
        for image in branch_images
    ]
)

In [ ]:
rel_energies = np.array([image.rel_energy_eV for image in raw_images])
saddle_image = int(np.argmax(rel_energies))
first_branch_dipole = branch_images[0].branch_dipole_eA
projected_delta = np.array(
    [(image.branch_dipole_eA - first_branch_dipole) @ field_direction for image in branch_images],
    dtype=float,
)
candidate_indices = candidate_indices_for_mode(BARRIER_MODE, len(raw_images), saddle_image)
requirements, impossible = coercive_requirements(
    rel_energies,
    projected_delta,
    candidate_indices,
    COERCIVE_CRITERION,
)
ec, limiting_image = coercive_field_from_requirements(
    requirements,
    impossible,
    COERCIVE_CRITERION,
    BARRIER_MODE,
    notes,
)

print("saddle_image =", saddle_image)
print("projected_delta =", projected_delta)
print("requirements =", requirements)
print("impossible =", impossible)
print("Ec =", ec)
print("limiting_image =", limiting_image)
print("notes =", notes)

In [ ]:
result = analyze_path(
    label=LABEL,
    path=PATH,
    energy_source=ENERGY_SOURCE,
    field_direction_arg=FIELD_DIRECTION_ARG,
    field_coordinates=FIELD_COORDINATES,
    barrier_mode=BARRIER_MODE,
    coercive_criterion=COERCIVE_CRITERION,
    initial_polarization=INITIAL_POLARIZATION,
    final_polarization_arg=FINAL_POLARIZATION,
    continuity_threshold_uC_cm2=CONTINUITY_THRESHOLD,
)

print("Ec =", result.coercive_field_MV_cm)
print("limiting_image =", result.limiting_image)
print("notes =", result.notes)

In [ ]:
rows = []
for label, path in default_samples(PATH):
    result = analyze_path(
        label=label,
        path=path,
        energy_source=ENERGY_SOURCE,
        field_direction_arg=FIELD_DIRECTION_ARG,
        field_coordinates=FIELD_COORDINATES,
        barrier_mode=BARRIER_MODE,
        coercive_criterion=COERCIVE_CRITERION,
        initial_polarization=INITIAL_POLARIZATION,
        final_polarization_arg=FINAL_POLARIZATION,
        continuity_threshold_uC_cm2=CONTINUITY_THRESHOLD,
    )
    rows.append(
        {
            "label": label,
            "path": str(path),
            "Ec": result.coercive_field_MV_cm,
            "limiting_image": result.limiting_image,
            "notes": " | ".join(result.notes),
        }
    )

pd.DataFrame(rows)